## 第27章　逻辑回归 —— 概率视角的分类器> 本章目标：用 Sigmoid 将线性输出压缩为概率，用交叉熵损失（第 19 章）优化二分类器。手动推导 ∂L/∂w 并用梯度下降训练，画出决策边界。理解为什么分类用交叉熵而不是 MSE——梯度信号的本质差异。> 前置知识：第 19 章（交叉熵）、第 22 章（softmax/sigmoid）、第 26 章（线性回归）---### 27.1　Sigmoid 函数 —— 把任意实数映射到 (0,1)线性回归输出的是任意实数值，但分类需要概率（0 到 1 之间）。**Sigmoid** 是那座桥梁：$$\sigma(z) = \frac{1}{1 + e^{-z}}$$当 z→+∞ 时 σ→1，当 z→−∞ 时 σ→0，在 z=0 处 σ=0.5——完美地将 ℝ 压缩到 (0,1)。📐 **定义　Sigmoid**：σ(z) = 1/(1+e⁻ᶻ)。单调递增，输出 ∈ (0,1)。导数 σ'(z) = σ(z)(1−σ(z))。二分类时等价于 softmax 的特例。💻 **代码　Sigmoid 曲线 + 导数**

In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef sigmoid(z):    return 1 / (1 + np.exp(-z))z = np.linspace(-8, 8, 200)s = sigmoid(z)ds = s * (1 - s)  # sigmoid derivativefig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(z, s, 'b-', lw=2); axes[0].axhline(y=0.5, color='gray', ls='--')axes[0].set_title('Sigmoid σ(z)'); axes[0].set_xlabel('z'); axes[0].grid(alpha=0.3)axes[1].plot(z, ds, 'r-', lw=2)axes[1].set_title("Sigmoid derivative σ'(z)=σ(1−σ)"); axes[1].set_xlabel('z'); axes[1].grid(alpha=0.3)plt.tight_layout(); plt.show()print("σ(0)=0.5, σ'(0)=0.25 (最大)")print("σ(5)≈1, σ'(5)≈0 (饱和→梯度消失)")

---### 27.2　前向传播 —— 从线性输出到分类概率逻辑回归的前向传播 = 线性变换 + sigmoid。`z = X @ w` 得到线性输出，`p = sigmoid(z)` 转为概率。预测类别：p≥0.5 → class 1，p<0.5 → class 0。💻 **代码　逻辑回归前向传播三行核心**

In [ ]:
import numpy as npnp.random.seed(42)n, d = 100, 2X = np.column_stack([np.random.randn(n, d), np.ones(n)])  # 加 bias 列w = np.random.randn(d + 1) * 0.1                          # 含 bias 的权重z = X @ w                              # 线性输出: (100,)p = 1 / (1 + np.exp(-z))               # sigmoid → 概率: (100,)y_pred = (p >= 0.5).astype(int)        # 阈值 0.5 → 类别print(f"z 范围: [{z.min():.2f}, {z.max():.2f}]")print(f"p 范围: [{p.min():.3f}, {p.max():.3f}]")print(f"预测类别分布: 0={np.sum(y_pred==0)}, 1={np.sum(y_pred==1)}")

> **关键洞察**：逻辑回归 = 一个没有隐藏层的神经网络。`X @ w` 是线性层，sigmoid 是激活函数。把这个"神经元"复制 256 份 + 再加一层，就是第 28 章的两层网络。---### 27.3　损失函数 —— 为什么必须用交叉熵分类不用 MSE 的原因：MSE 的梯度含 σ'(z) 因子——当预测接近 0 或 1 时 σ'≈0，梯度消失，模型停止学习。交叉熵的梯度 = (σ(z)−y)·x——梯度大小正比于预测误差，完美避开饱和问题。📐 **交叉熵损失（二分类）**：L = −[y·log(p) + (1−y)·log(1−p)]。梯度 ∂L/∂w = (p−y)·x。💻 **代码　交叉熵 vs MSE 梯度对比**

In [ ]:
# 模拟一个"预测很自信但错了"的场景y_true = 1.0p_wrong = 0.01  # 模型预测 1% 概率（非常确定是负类——猜错了）# MSE: L = (p - y)², 梯度 = 2(p - y) * p(1-p)mse_grad = 2 * (p_wrong - y_true) * p_wrong * (1 - p_wrong)# 交叉熵: L = -[y*log(p) + (1-y)*log(1-p)], 梯度 = p - yce_grad = p_wrong - y_trueprint(f"预测 p=0.01, 真实 y=1（模型大错特错）:")print(f"  MSE 梯度:     {abs(mse_grad):.6f}  ← 饱和！σ'(z)≈0 截断了梯度")print(f"  交叉熵梯度:   {abs(ce_grad):.6f}    ← 梯度大小 = 预测误差")print(f"  交叉熵/MSE:   {abs(ce_grad/mse_grad):.0f}x → 交叉熵远更善于纠正错误")

> **关键洞察**：交叉熵的梯度不含 σ'(z) 因子——这是数学推导的结果（sigmoid + log 的导数恰好消掉了 σ'）。第 19 章从信息论证明了这不是巧合：交叉熵是分类问题的最优损失函数。---### 27.4　手动反向传播 —— 推导 ∂L/∂w 并更新链式法则（第 13 章）：∂L/∂w = ∂L/∂p · ∂p/∂z · ∂z/∂w。经历 sigmoid + 交叉熵联合求导后，得到简洁结果：∂L/∂w = (p−y)·x——预测值与真实值的差乘以输入。💻 **代码　手动反向传播 + 参数更新（3 行核心逻辑）**

In [ ]:
# 模拟一小批数据X_batch = np.random.randn(4, d + 1); X_batch[:, -1] = 1  # biasy_batch = np.array([0, 1, 0, 1])w_batch = np.random.randn(d + 1) * 0.1# 前向p_batch = 1 / (1 + np.exp(-X_batch @ w_batch))# 反向：交叉熵+sigmoid 的联合梯度 = (p - y) @ Xgrad = X_batch.T @ (p_batch - y_batch) / len(y_batch)# 更新lr = 0.1w_batch -= lr * grad# 验证梯度公式：数值梯度eps = 1e-5grad_num = np.zeros_like(w_batch)for i in range(len(w_batch)):    w_plus = w_batch.copy(); w_plus[i] += eps    p_plus = 1 / (1 + np.exp(-X_batch @ w_plus))    loss_plus = -np.mean(y_batch * np.log(p_plus + 1e-12) + (1-y_batch) * np.log(1-p_plus + 1e-12))    w_minus = w_batch.copy(); w_minus[i] -= eps    p_minus = 1 / (1 + np.exp(-X_batch @ w_minus))    loss_minus = -np.mean(y_batch * np.log(p_minus + 1e-12) + (1-y_batch) * np.log(1-p_minus + 1e-12))    grad_num[i] = (loss_plus - loss_minus) / (2 * eps)print(f"解析梯度 ∂L/∂w: {grad.round(5)}")print(f"数值梯度 ∂L/∂w: {grad_num.round(5)}")print(f"两者一致: {np.allclose(grad, grad_num, atol=1e-5)} ✓")

> **关键洞察**：∂L/∂w = (p−y)·x 的优雅简洁不是巧合——sigmoid 的导数 σ'(z)=σ(z)(1−σ(z)) 恰好与交叉熵的 1/p 和 1/(1−p) 因子抵消。这个"巧合"是数学上选择 sigmoid+交叉熵搭配的原因。🔗 **AI 连接**：第 28 章的两层网络反向传播是这个公式的矩阵扩展——多了一层 ReLU + 权重矩阵，但底层逻辑完全一样：链式法则 → 本地梯度 → 乘以上游梯度。---### 27.5　决策边界可视化💻 **代码　逻辑回归完整训练 + 决策边界**

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)n = 100# 两类数据X0 = np.random.randn(n//2, 2) * 0.7 + np.array([-2, -1])X1 = np.random.randn(n//2, 2) * 0.7 + np.array([2, 1])X_data = np.vstack([X0, X1])X = np.column_stack([X_data, np.ones(n)])  # add biasy = np.array([0]*(n//2) + [1]*(n//2))def sigmoid(z):    return 1 / (1 + np.exp(-z))# 梯度下降w = np.zeros(3)lr = 0.1for _ in range(500):    p = sigmoid(X @ w)    w -= lr * X.T @ (p - y) / n# 准确率preds = (sigmoid(X @ w) > 0.5).astype(int)acc = (preds == y).mean()print(f"训练准确率: {acc:.1%}")# 决策边界: w[0]*x1 + w[1]*x2 + w[2] = 0 → x2 = -(w[0]*x1+w[2])/w[1]x_grid = np.linspace(-4, 4, 100)y_boundary = -(w[0]*x_grid + w[2]) / w[1]plt.figure(figsize=(8, 6))plt.scatter(X0[:,0], X0[:,1], c='blue', alpha=0.6, label='Class 0')plt.scatter(X1[:,0], X1[:,1], c='red', alpha=0.6, label='Class 1')plt.plot(x_grid, y_boundary, 'k-', lw=2, label='Decision Boundary')plt.xlabel('x1'); plt.ylabel('x2')plt.title(f'Logistic Regression (acc={acc:.1%})')plt.legend(); plt.grid(alpha=0.3); plt.show()

> **关键洞察**：决策边界之所以是直线，因为 sigmoid 只改变了输出的"解释方式"（从数值变成概率），并没有改变模型的线性本质。第 28 章的两层网络引入 ReLU 非线性后，决策边界可以弯曲——这是从线性模型到非线性模型的关键跨越。🔗 **AI 连接**：逻辑回归是理解"分类"的最小完整模型——前向（sigmoid）、损失（交叉熵）、反向（手推梯度）三要素俱全。第 28 章的两层网络只是把这个逻辑复制了两遍。---**✏️ 习题**1. （概念）为什么分类用交叉熵而不是 MSE？从梯度的角度解释。2. （代码）生成同心圆数据（`sklearn.datasets.make_circles`），用逻辑回归分类。观察决策边界为什么无法分离同心圆——引出对非线性模型的需求。3. （代码）手写逻辑回归的梯度下降训练循环。打印每 100 轮的 loss 和准确率，画出 loss 下降曲线。---> 🔗 **章末钩子**：逻辑回归只能画直线。要让决策边界弯曲，需要引入非线性激活函数——把多层变换"扭"成曲线。> 预览：下一章——**两层神经网络**，NumPy 手写反向传播。